## 🎯 Learning Objectives
* Understand the limitations of single-vector retrieval for complex queries.
* Grasp the concept of multi-vector retrieval and its benefits in RAG systems.
* Learn how Hypothetical Document Embeddings (HyDE) improve retrieval by generating synthetic documents.
* Implement multi-vector retrieval and HyDE using modern RAG frameworks and tools.
* Evaluate the performance characteristics and trade-offs of these advanced retrieval techniques.


## Multi-vector Retrieval and Hypothetical Document Embeddings (HyDE): Elevating RAG Precision

In the rapidly evolving landscape of Retrieval-Augmented Generation (RAG), the quality of retrieved documents directly dictates the quality of the generated answer. While basic vector search has been a cornerstone, complex queries and nuanced information often expose its limitations. This lesson dives into two advanced techniques – **Multi-vector Retrieval** and **Hypothetical Document Embeddings (HyDE)** – that significantly enhance retrieval precision and recall, crucial for building state-of-the-art agentic RAG systems.

### The Challenge: Beyond Single-Vector Simplicity

Imagine you're searching for a book. A single-vector search is like looking for a book solely by its exact title. If your query is slightly off, or you're looking for a book based on its plot, genre, or author, a simple title search might fail. Similarly, in RAG, a single embedding for a document chunk might not capture all its facets. A query might be about a specific detail, while the chunk's primary embedding focuses on its overall theme. This 'semantic mismatch' or 'lexical gap' often leads to suboptimal retrieval.

### Multi-vector Retrieval: A Multi-faceted View of Information

Multi-vector retrieval addresses this by representing a single logical document or chunk with *multiple* embeddings, each capturing a different aspect or granularity. Think of it as indexing a book not just by its title, but also by its summary, its key characters, its genre, and even individual chapter summaries. When a query comes in, it can match against any of these diverse representations, leading to a more robust and comprehensive search.

**How it works:**
1.  **Document Decomposition:** Break down large documents into smaller, manageable chunks (e.g., paragraphs, sentences).
2.  **Multi-representation Generation:** For each chunk, generate several different representations:
    *   **Summary:** A concise summary of the chunk, often generated by an LLM.
    *   **Full Text:** The original text of the chunk.
    *   **Question-Answer Pairs:** Generate hypothetical Q&A pairs that the chunk could answer.
    *   **Keywords/Entities:** Extract key terms or entities.
3.  **Embedding:** Embed each of these representations using a suitable embedding model.
4.  **Indexing:** Store all these embeddings in a vector database. Crucially, each 'sub-embedding' (e.g., summary embedding, Q&A embedding) is linked back to the *original, larger document chunk* it represents.
5.  **Retrieval:** When a query arrives, it's embedded and used to search against *all* these diverse embeddings. If any of the sub-embeddings match, the original, full document chunk is retrieved.

**Benefits:**
*   **Improved Recall:** Catches relevant documents even if the query doesn't directly match the main text embedding.
*   **Handles Diverse Queries:** Better at answering queries that are abstract, specific, or multi-intent.
*   **Granular Control:** Allows for fine-grained retrieval based on different aspects of the document.

### Hypothetical Document Embeddings (HyDE): Bridging the Semantic Gap

HyDE takes a different, yet complementary, approach to improving retrieval. The core idea is to bridge the 'semantic gap' between a short, often vague query and a potentially long, detailed document. Instead of directly embedding the query and searching, HyDE first uses a Large Language Model (LLM) to *generate a hypothetical document* that would answer the user's query. This hypothetical document is typically longer and more descriptive than the original query, making its embedding semantically closer to actual relevant documents.

**How it works:**
1.  **User Query:** The user submits a query (e.g., "What are the benefits of quantum computing?").
2.  **Hypothetical Document Generation:** An LLM (e.g., GPT-4o, Claude 3.5 Sonnet) is prompted to generate a plausible, but potentially non-factual, document that *would answer* the query. For example, it might generate a paragraph explaining the benefits of quantum computing.
3.  **Hypothetical Document Embedding:** This generated hypothetical document is then embedded using the same embedding model used for the actual documents in your vector store.
4.  **Vector Search:** The embedding of the *hypothetical document* is used to perform a similarity search against your vector database of actual documents.
5.  **Final RAG:** The retrieved actual documents, along with the original query, are then passed to another LLM for final answer generation.

**Analogy:** Imagine you ask a librarian, "Tell me about that book with the blue cover and a dragon on it." Instead of searching for "blue cover dragon," the librarian (LLM) *guesses* you might be looking for a fantasy novel about a dragon rider, then searches for books matching that *guessed description*. Even if the guess isn't perfectly accurate, it's often a much better search query than your initial vague one.

**Benefits:**
*   **Overcomes Lexical Mismatch:** Excellent for queries that use different terminology than the documents.
*   **Handles Vague Queries:** LLMs can infer intent and generate more descriptive search queries.
*   **Improved Semantic Alignment:** The hypothetical document's embedding is often more semantically aligned with actual documents than the short query's embedding.

### Combining Forces: A Synergistic Approach

Multi-vector retrieval and HyDE are not mutually exclusive; they can be combined for even greater effect. An agentic RAG system might use HyDE to generate a robust query embedding, which then searches against a multi-vector index, leveraging the strengths of both techniques to achieve unparalleled retrieval accuracy. This is particularly powerful in LangGraph, where different retrieval strategies can be orchestrated as distinct nodes within a complex agentic workflow.


In [ ]:
# Ensure you have the necessary libraries installed. As of 2026, LangChain and its ecosystem are mature.
# pip install langchain langchain-community langchain-openai chromadb sentence-transformers tiktoken

import os
from typing import List, Dict

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.retrievers import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter

# --- Configuration --- #
# Set your OpenAI API key. In a production environment, use environment variables or a secure secret manager.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize LLM and Embedding Model
# Using a modern, efficient model for generation and embeddings.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1) # gpt-4o-mini is a good balance of cost/performance for HyDE
embeddings = OpenAIEmbeddings(model="text-embedding-3-small") # text-embedding-3-small for efficiency

# --- Sample Documents --- #
# In a real-world scenario, these would come from your data source.
docs = [
    Document(page_content="Quantum computing harnesses quantum-mechanical phenomena like superposition and entanglement to solve problems intractable for classical computers. Its potential applications include drug discovery, materials science, and cryptography.", metadata={"source": "wiki", "topic": "quantum_computing"}),
    Document(page_content="The latest advancements in AI agents involve sophisticated planning, tool use, and self-correction mechanisms. Frameworks like LangGraph enable developers to build complex, stateful agentic workflows, moving beyond simple chain-based interactions.", metadata={"source": "blog", "topic": "ai_agents"}),
    Document(page_content="Blockchain technology, foundational to cryptocurrencies like Bitcoin and Ethereum, provides a decentralized and immutable ledger. Smart contracts, self-executing agreements stored on a blockchain, are revolutionizing finance and supply chain management.", metadata={"source": "report", "topic": "blockchain"}),
    Document(page_content="The development of advanced robotics is leading to breakthroughs in autonomous navigation and human-robot interaction. Collaborative robots (cobots) are increasingly deployed in manufacturing, working alongside humans to improve efficiency and safety.", metadata={"source": "journal", "topic": "robotics"}),
    Document(page_content="Artificial General Intelligence (AGI) remains a long-term goal for AI research, aiming to create machines with human-level cognitive abilities across a wide range of tasks. Current efforts focus on scaling foundation models and developing robust reasoning capabilities.", metadata={"source": "research", "topic": "agi"})
]

# --- Multi-vector Retrieval Implementation --- #

# 1. Define a text splitter for chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

# 2. Create a document store (in-memory for this example, but could be a database)
# This store holds the full, original documents/chunks.
store = InMemoryStore()

# 3. Initialize the vector store for child chunks (summaries, Q&A, etc.)
# We'll use ChromaDB, a popular choice for local development and production.
vectorstore = Chroma(collection_name="multi_vector_retrieval", embedding_function=embeddings)

# 4. Create the MultiVectorRetriever
# The 'id_key' links child chunks back to their parent in the InMemoryStore.
retriever = MultiVectorRetriever(vectorstore=vectorstore, docstore=store, id_key="doc_id")

# 5. Process documents for multi-vector indexing
# For simplicity, we'll generate summaries as an additional vector representation.
# In a real system, you might generate Q&A pairs or other representations.

def generate_summary(doc_content: str) -> str:
    """Generates a concise summary of the document content using an LLM."""
    prompt = f"Summarize the following text concisely, focusing on key concepts:\n\n{doc_content}\n\nSummary:"
    response = llm.invoke(prompt)
    return response.content

print("Indexing documents for multi-vector retrieval...")
for i, doc in enumerate(docs):
    doc_id = f"doc_{i}"
    
    # Store the original document in the docstore
    store.mset([(doc_id, doc)])
    
    # Generate a summary for the document
    summary = generate_summary(doc.page_content)
    summary_doc = Document(page_content=summary, metadata=doc.metadata)
    
    # Add the summary and the original content as 'child' documents to the vectorstore
    # The 'doc_id' metadata links them back to the original document in the docstore.
    retriever.vectorstore.add_documents([summary_doc, doc], ids=[f"{doc_id}_summary", f"{doc_id}_full"], metadatas=[{"doc_id": doc_id}, {"doc_id": doc_id}])

print("Multi-vector indexing complete.\n")

# --- HyDE Implementation --- #

def generate_hyde_document(query: str) -> str:
    """Generates a hypothetical document that would answer the given query."""
    prompt = f"Generate a concise, hypothetical document that would answer the following question:\n\nQuestion: {query}\n\nHypothetical Document:"
    response = llm.invoke(prompt)
    return response.content

# --- Example Usage: Combining HyDE with Multi-vector Retrieval --- #

query = "What are the latest breakthroughs in AI agents and how do they relate to complex workflows?"

print(f"Original Query: '{query}'")

# 1. Generate hypothetical document using HyDE
hyde_doc_content = generate_hyde_document(query)
print(f"\nHypothetical Document (HyDE):\n{hyde_doc_content}\n")

# 2. Embed the hypothetical document
hyde_embedding = embeddings.embed_query(hyde_doc_content)

# 3. Use the HyDE embedding to query the multi-vector retriever
# We need to directly query the vectorstore with the HyDE embedding
# and then use the retriever's docstore to get the full documents.

# Perform similarity search using the HyDE embedding
# We'll retrieve the top 2 most similar child documents (summaries or full texts)
retrieved_child_docs = vectorstore.similarity_search_by_vector(hyde_embedding, k=2)

print("Retrieved child documents (from vectorstore, based on HyDE embedding):")
for doc in retrieved_child_docs:
    print(f"- Content: {doc.page_content[:100]}... (Source: {doc.metadata.get('source')}, Type: {'summary' if 'summary' in doc.id else 'full'}) ")

# Now, retrieve the full parent documents using the doc_ids from the child documents
# The MultiVectorRetriever handles this abstraction if you use its .invoke() method.
# For demonstration, we'll manually extract and retrieve.

# Extract unique parent doc_ids from the retrieved child documents
parent_doc_ids = list(set([doc.metadata['doc_id'] for doc in retrieved_child_docs if 'doc_id' in doc.metadata]))

final_retrieved_docs = []
for doc_id in parent_doc_ids:
    parent_doc = store.mget([doc_id])
    if parent_doc and parent_doc[0]:
        final_retrieved_docs.append(parent_doc[0])

print("\nFinal Retrieved Parent Documents (from docstore):")
for i, doc in enumerate(final_retrieved_docs):
    print(f"--- Document {i+1} ---")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Topic: {doc.metadata.get('topic')}")
    print(f"Content: {doc.page_content[:200]}...")
    print("--------------------\n")

# In a LangGraph agent, this retrieval step would be a node,
# passing `final_retrieved_docs` to the next node for synthesis.


### Interpreting the Output and Performance Considerations

The code demonstrates a practical implementation of multi-vector retrieval combined with HyDE. Let's break down what the output signifies and discuss the implications for real-world RAG systems.

**Interpreting the Output:**
1.  **Hypothetical Document (HyDE):** You'll first see the LLM-generated hypothetical document. Notice how it expands on the original query, providing more context and detail. This expanded text is what gets embedded, allowing for a more semantically rich search against your vector store.
2.  **Retrieved Child Documents:** These are the actual entries from your vector database that were most similar to the HyDE embedding. In our example, these could be either the generated summaries or the full text chunks, each linked back to its original parent document via the `doc_id` metadata. The output shows a snippet of their content and their type (summary or full).
3.  **Final Retrieved Parent Documents:** This is the crucial output. Based on the `doc_id`s from the matched child documents, the system retrieves the *full, original parent documents* from the `InMemoryStore`. These are the documents that would then be passed to your RAG LLM for synthesizing the final answer. You should observe that the retrieved documents are highly relevant to the original query, even if the query itself was somewhat abstract or used different phrasing than the document content.

**Performance Trade-offs and Considerations (2026 Context):**

**Benefits:**
*   **Significantly Improved Recall and Precision:** Both techniques excel at finding relevant information that might be missed by simpler retrieval methods. HyDE helps overcome the 'query-document semantic gap,' while multi-vector retrieval ensures different facets of a document are discoverable.
*   **Robustness to Query Variation:** Queries can be vague, use synonyms, or focus on specific aspects, and these methods are more likely to find a match.
*   **Enhanced User Experience:** Leads to more accurate and comprehensive answers from the RAG system, reducing instances of 'no answer found' or irrelevant responses.

**Costs and Challenges:**
*   **Increased Indexing Time and Storage (Multi-vector):** Generating multiple representations (summaries, Q&A pairs) for each document chunk and embedding them requires more computational resources during indexing and significantly more storage space in your vector database. This is a one-time cost, but scales with your corpus size.
*   **Increased Inference Time and Cost (HyDE):** The primary overhead of HyDE is the LLM call required to generate the hypothetical document *for every single query*. While `gpt-4o-mini` and similar efficient models (e.g., Anthropic's Haiku, Google's Gemini Flash) are much faster and cheaper than their predecessors, this still adds latency and cost compared to direct query embedding. For high-throughput systems, this needs careful optimization (e.g., caching, parallelization).
*   **LLM Hallucination Risk (HyDE):** The hypothetical document generated by the LLM might contain non-factual information. However, this is generally not a critical issue for retrieval, as its purpose is solely to generate a better *search vector*, not to provide factual content to the user. The final answer is still generated from the *actual* retrieved documents.
*   **System Complexity:** Implementing and managing multi-vector indexes and HyDE adds layers of complexity to your RAG architecture. This includes managing different embedding types, ensuring proper linking between child and parent documents, and orchestrating the LLM calls for HyDE.
*   **Embedding Model Choice:** The quality of your embedding model is paramount. As of 2026, highly performant and specialized embedding models (e.g., `text-embedding-3-large`, various open-source models fine-tuned for specific domains) are readily available, but choosing the right one for your data is crucial.

**Typical Use Cases:**
*   **Complex Enterprise Search:** Where users ask highly specific or abstract questions across vast, diverse document repositories (e.g., legal, medical, technical documentation).
*   **Customer Support Bots:** Handling ambiguous or multi-intent customer queries more effectively.
*   **Research and Development:** Assisting researchers in finding highly relevant papers or data points from scientific literature.
*   **Any RAG system where recall and precision are critical** and the cost/latency overhead is acceptable for the improved answer quality.

In LangGraph, these advanced retrieval techniques would typically be encapsulated within a dedicated node or subgraph. An agent could dynamically decide whether to employ HyDE based on query complexity, or use a multi-vector retriever as its default search mechanism, enabling highly adaptive and intelligent information retrieval workflows.


### Resources for Further Exploration

To deepen your understanding and implementation of multi-vector retrieval and HyDE, explore the following resources:

*   **LangChain Documentation on Retrievers:**
    *   [Parent Document Retriever (Multi-vector concept)](https://python.langchain.com/docs/modules/data_connection/retrievers/parent_document_retriever)
    *   [MultiVectorRetriever (Advanced Multi-vector)](https://python.langchain.com/docs/modules/data_connection/retrievers/multi_vector)
*   **Original HyDE Paper:**
    *   [HyDE: Hypothetical Document Embeddings for Improved Zero-Shot Information Retrieval](https://arxiv.org/abs/2212.10496)
*   **Hugging Face Transformers:**
    *   Explore various embedding models and LLMs for HyDE generation: [Hugging Face Models](https://huggingface.co/models)
*   **Vector Database Documentation:**
    *   **Chroma:** [ChromaDB Documentation](https://docs.trychroma.com/)
    *   **Weaviate:** [Weaviate Documentation](https://weaviate.io/developers/weaviate/current)
    *   **Pinecone:** [Pinecone Documentation](https://docs.pinecone.io/)
    *   **Qdrant:** [Qdrant Documentation](https://qdrant.tech/documentation/)
*   **LangGraph Documentation:**
    *   Learn how to integrate custom retrieval strategies into agentic workflows: [LangGraph Introduction](https://langchain-ai.github.io/langgraph/)
